# ch05 Bonus 05：Muon 优化器

> 对照官方 `ch05/18_muon`
> **参考**：Moonshot / Keller Jordan 2024 的 Muon 优化器

## 一句话

对二维参数（权重矩阵）的动量做**牛顿-舒尔茨正交化**，让每步更新方向更均衡，收敛更快更稳。

## 核心思想

AdamW 对每个参数维度独立调整步长。Muon 进一步：对矩阵参数的动量矩阵做正交化（让奇异值均匀化），避免某些方向更新过快、某些过慢。

**牛顿-舒尔茨迭代**（5 步近似矩阵符号函数/正交化）：
```
X = G / ||G||
重复 5 次： X = aX + b(A)X + c(A²)X,  其中 A = XXᵀ
```
常数 a=3.4445, b=-4.7750, c=2.0315 是论文给的最优系数。

> Muon 在小模型上效果显著（训练步数减少 2-3×），但对 embedding/1D 参数仍用 AdamW。

In [ ]:
import torch
import torch.nn as nn


def newton_schulz(G, steps=5):
    """牛顿-舒尔茨正交化：把矩阵 G 变成近正交矩阵。"""
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    return X


class Muon(torch.optim.Optimizer):
    """Muon 优化器教学版：对 2D 参数用正交化动量。"""

    def __init__(self, params, lr=0.02, momentum=0.95):
        defaults = dict(lr=lr, momentum=momentum)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            mu = group["momentum"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p.grad)
                buf = state["momentum_buffer"]
                # 动量累积
                buf.mul_(mu).add_(p.grad)
                if p.dim() >= 2:
                    # 关键：对 2D 矩阵参数做正交化
                    update = newton_schulz(buf)
                    # 按矩阵维度缩放（Muon 论文的经验比例）
                    update *= max(1, p.shape[-2] / p.shape[-1]) ** 0.5
                else:
                    # 1D 参数（bias/norm）用普通动量
                    update = buf
                p.add_(update, alpha=-lr)

In [ ]:
# 验证牛顿-舒尔茨正交化效果
torch.manual_seed(42)
G = torch.randn(64, 32)
G_ortho = newton_schulz(G)
# 正交化后 G_ortho @ G_ortho.T 应接近单位阵
ortho_err = ((G_ortho @ G_ortho.T) - torch.eye(64)).abs().mean().item()
print(f"正交化误差: {ortho_err:.4f}（越小越正交，应 < 0.1）")

# 对比 Muon vs AdamW 在一个小问题上的收敛
torch.manual_seed(0)
target = torch.randn(64, 32)

def bench(opt_class, lr, steps=50):
    torch.manual_seed(0)
    W = torch.randn(64, 32, requires_grad=True)
    opt = opt_class([W], lr=lr)
    losses = []
    for _ in range(steps):
        opt.zero_grad()
        loss = ((W - target) ** 2).sum()
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

adam_losses = bench(torch.optim.AdamW, lr=0.1)
muon_losses = bench(Muon, lr=0.05)
print(f"\n收敛 50 步后损失:")
print(f"  AdamW: {adam_losses[-1]:.4f}")
print(f"  Muon:  {muon_losses[-1]:.4f}  ← 通常更快")
print("\n💡 Muon 在矩阵参数优化上常比 AdamW 快 2-3×。")